Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Optimización de hiperparámetros con *bayesian optimization***

Vamos a trabajar con el mismo *data set*: `spambase.csv`.

Recordemos que el mismo es una modificación del data set publicado en el siguiente [enlace](https://www.kaggle.com/datasets/somesh24/spambase). Este contiene registros de *emails* junto a si son clasificados como *spam* (1) o no (0). Los atributos representan la frecuencia de aparición de ciertas palabras, caracteres y mayúsculas. Para más información al respecto, visitar el enlace antes adjunto.

In [ ]:
import pandas as pd

data = pd.read_csv('spambase.csv')

### **Instalación de HyperOpt**

In [ ]:
!pip install hyperopt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.3 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.6 MB/s eta 0:00:00


### **Separación de predictores y etiquetas**

In [ ]:
X = data.drop('Class', axis = 'columns')
y = data['Class']

### **Bayesian optimization con HyperOpt**

La librería HyperOpt es de las más utilizadas en Python que implementan la técnica en cuestión.

##### **Definición de la función de costo**

La misma debe devolver un real que el algoritmo buscará **minimizar**.

En este caso, dado que el método `score` para una instancia de `DecisionTreeClassifier` calcula la métrica `accuracy` (que se desea maximizar), haremos que retorne **1 - accuracy**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from hyperopt import STATUS_OK

def objective(params):
    tree = DecisionTreeClassifier(**params, random_state = 22)
    score = cross_val_score(tree, X, y, cv = KFold(4)).mean() # Aplicamos validación cruzada con 4 folds.
    return {'loss': 1 - score, 'status': STATUS_OK}

##### **Definición del espacio de hiperparámetros**

Para ello, usamos el modulo `hp` de HyperOpt.

Los hiperparámetros que vamos a considerar y sus distribuciones son:
- `criterion`: {gini, entropy, log_loss};
- `splitter`: {best, random};
- `max_depth`: uniform[3, 80] (discreta);
- `min_samples_split`: uniform[2, 20] (discreta);
- `min_samples_leaf`: uniform[1, 20] (discreta) y
- `min_impurity_decrease`: uniform[0, 0.1] (continua).

In [ ]:
from hyperopt import hp

space = {'criterion': hp.choice('criterion', ['gini', 'entropy', 'log_loss']),
         'splitter': hp.choice('splitter', ['best', 'random']),
         'max_depth': hp.uniformint('max_depth', 3, 80),
         'min_samples_split': hp.uniformint('min_samples_split', 2, 20),
         'min_samples_leaf': hp.uniformint('min_samples_leaf', 1, 20),
         'min_impurity_decrease': hp.uniform('min_impurity_decrease', 0, 0.1)
        }

##### **Optimización**

**TPE (Tree Parzen Estimator)** es el estimador que decide qué combinación de hiperparámetros probar, en cada iteración, en base al aprendizaje previo.

In [ ]:
from hyperopt import fmin, tpe, space_eval
import numpy as np

best = fmin(objective, space,
            algo = tpe.suggest,
            max_evals = 1800,
            rstate = np.random.default_rng(22))

100%|██████████| 1800/1800 [05:01<00:00,  5.97trial/s, best loss: 0.1343478260869565] 


El score resultante equivale a 0,866 aproximadamente.

Ese valor de score es mayor al ≈ 0,847 que obtuvimos antes con *grid search* y al ≈ 0,859 que obtuvimos antes con *random search*.

Notemos que dicho score fue resultado de asignar a HyperOpt la misma cantidad de iteraciones que en los experimentos anteriores (1.800). Veamos entonces, para los casos en que no tenemos suficiente tiempo o poder de cómputo, qué sucede cuando se le asigna a HyperOpt un menor número de iteraciones. En este caso, usaremos **200**.

In [ ]:
best = fmin(objective, space,
            algo = tpe.suggest,
            max_evals = 200,
            rstate = np.random.default_rng(22))

100%|██████████| 200/200 [00:23<00:00,  8.62trial/s, best loss: 0.13630434782608702]


In [ ]:
params = space_eval(space, best) # Guardamos los hiperparámetros ganadores.
print(params)

{'criterion': 'log_loss', 'max_depth': 32, 'min_impurity_decrease': 0.0029068130133325133, 'min_samples_leaf': 9, 'min_samples_split': 7, 'splitter': 'best'}


En pocos segundos de búsqueda, se llega a un score de ≈ **0,8637** (1 - 0,1363...), el cual es superior a los obtenidos con grid search y random search. Incluso, la diferencia entre este score y el obtenido previamente mediante el mismo algoritmo, pero con 1.800 iteraciones, es inapreciable.

Por lo tanto, a los fines prácticos, **bayesian optimization** resulta una herramienta ideal para obtener una buena configuración de hiperparámetros, tanto por su poco procesamiento requerido como por su importante *performance*.